# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [41]:
!cat publications.tsv

pub_date	category	title	venue	excerpt	citation	url_slug	paper_url	slides_url
08/12/2024	manuscripts	Enhancer heterogeneity in acute lymphoblastic leukemia drives differential gene expression between patients	bioRxiv	This study explores how enhancer variability in ALL patients contributes to divergent gene expression.		enhancer-heterogeneity-leukemia	https://doi.org/10.1101/2024.12.08.627394	
18/09/2024	manuscripts	The fetal specific gene LIN28B is essential for human fetal B-lymphopoiesis and initiation of KMT2A::AFF1 infant leukemia	bioRxiv	Shows that LIN28B is critical for fetal B-cell development and initiation of infant leukemia.		lin28b-infant-leukemia	https://doi.org/10.1101/2024.09.18.613730	
31/08/2023	manuscripts	MLL-AF4 cooperates with PAF1 and FACT to drive high density enhancer interactions in leukemia	Nature Communications	Describes how MLL-AF4 collaborates with co-factors to shape enhancer architecture in leukemia.		mll-af4-enhancer-interactions	https://doi.org/10.1038/s4

## Import pandas

We are using the very handy pandas library for dataframes.

In [42]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [43]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,category,title,venue,excerpt,citation,url_slug,paper_url,slides_url
0,08/12/2024,manuscripts,Enhancer heterogeneity in acute lymphoblastic ...,bioRxiv,This study explores how enhancer variability i...,NaN,enhancer-heterogeneity-leukemia,https://doi.org/10.1101/2024.12.08.627394,NaN
1,18/09/2024,manuscripts,The fetal specific gene LIN28B is essential fo...,bioRxiv,Shows that LIN28B is critical for fetal B-cell...,NaN,lin28b-infant-leukemia,https://doi.org/10.1101/2024.09.18.613730,NaN
2,31/08/2023,manuscripts,MLL-AF4 cooperates with PAF1 and FACT to drive...,Nature Communications,Describes how MLL-AF4 collaborates with co-fac...,NaN,mll-af4-enhancer-interactions,https://doi.org/10.1038/s41467-023-40981-9,NaN
3,15/06/2023,manuscripts,Endothelial sensing of AHR ligands regulates i...,Nature,Demonstrates how endothelial AHR sensing maint...,NaN,ahr-ligands-intestinal-homeostasis,https://doi.org/10.1038/s41586-023-06508-4,NaN
4,01/05/2021,manuscripts,Dysregulation of the Pdx1/Ovol2/Zeb2 axis in d...,Molecular Metabolism,Explores how loss of Pdx1 control in β-cells i...,NaN,pdx1-emtdiabetes,https://doi.org/10.1016/j.molmet.2021.101248,NaN
5,01/12/2024,conferences,Differential Gene Expression in KMT2A::AFF1 Le...,66th American Society of Hematology Annual Mee...,NaN,NaN,enhancer-heterogeneity-ash-2024,https://doi.org/10.1182/blood-2024-208035,NaN
6,01/12/2024,conferences,The fetal specific gene LIN28B is essential fo...,66th American Society of Hematology Annual Mee...,NaN,NaN,lin28b-fetal-leukemia,https://doi.org/10.1182/blood-2024-198819,NaN
7,01/05/2022,conferences,A high-throughput cell culture model of Idiopa...,25th Annual Meeting of the American Society of...,NaN,NaN,ipf-model-asgct,https://doi.org/10.1016/j.ymthe.2022.04.017,NaN
8,01/10/2024,posters,Identifying the DNA binding specificity of chr...,"Molecular Haemopoiesis, London, UK",NaN,NaN,dna-binding-chromatin-complexes,NaN,https://cchahrour.github.io/files/Molhaem_post...
9,01/06/2024,posters,DNA binding specificity of MLL-AF4 in leukemia,Genome Regulation and Cellular Fates in Homeos...,NaN,NaN,mll-af4-binding,NaN,https://cchahrour.github.io/files/Gen_reg_post...


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [44]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [45]:
import os
from html import escape as html_escape

for row, item in publications.iterrows():
    # Format date to ISO: YYYY-MM-DD
    iso_date = item.pub_date
    if "/" in iso_date:
        day, month, year = iso_date.split("/")
        iso_date = f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    md_filename = f"{iso_date}-{item.url_slug}.md"
    html_filename = f"{iso_date}-{item.url_slug}"

    # Start YAML frontmatter
    md = f"---\ntitle: \"{html_escape(item.title)}\"\n"
    md += "collection: publications"
    md += f"\npermalink: /publication/{html_filename}"
    
    if isinstance(item.excerpt, str) and len(item.excerpt) > 5:
        md += f"\nexcerpt: '{html_escape(item.excerpt)}'"
    
    md += f"\ndate: {iso_date}"
    md += f"\nvenue: '{html_escape(item.venue)}'"
    md += f"\ncategory: {item.category}"

    if isinstance(item.slides_url, str) and len(item.slides_url) > 5:
        md += f"\nslidesurl: '{item.slides_url}'"
    if isinstance(item.paper_url, str) and len(item.paper_url) > 5:
        md += f"\npaperurl: '{item.paper_url}'"
    if isinstance(item.citation, str) and len(item.citation) > 5:
        md += f"\ncitation: '{html_escape(item.citation)}'"

    md += "\n---\n"
        
    # Save markdown file
    output_path = os.path.join("../_publications", os.path.basename(md_filename))
    with open(output_path, "w") as f:
        f.write(md)


These files are in the publications directory, one directory below where we're working from.

In [46]:
!ls ../_publications/

2021-05-01-pdx1-emtdiabetes.md
2022-05-01-ipf-model-asgct.md
2023-06-15-ahr-ligands-intestinal-homeostasis.md
2023-08-31-mll-af4-enhancer-interactions.md
2024-06-01-mll-af4-binding.md
2024-09-18-lin28b-infant-leukemia.md
2024-10-01-dna-binding-chromatin-complexes.md
2024-12-01-enhancer-heterogeneity-ash-2024.md
2024-12-01-lin28b-fetal-leukemia.md
2024-12-08-enhancer-heterogeneity-leukemia.md


In [47]:
!cat ../_publications/2021-05-01-pdx1-emtdiabetes.md

---
title: "Dysregulation of the Pdx1/Ovol2/Zeb2 axis in dedifferentiated β-cells triggers the induction of genes associated with epithelial-mesenchymal transition in diabetes"
collection: publications
permalink: /publication/2021-05-01-pdx1-emtdiabetes
excerpt: 'Explores how loss of Pdx1 control in β-cells induces EMT-like gene expression in diabetes.'
date: 2021-05-01
venue: 'Molecular Metabolism'
category: manuscripts
paperurl: 'https://doi.org/10.1016/j.molmet.2021.101248'
---


In [48]:
!cat ../_publications/2024-06-01-mll-af4-binding.md

---
title: "DNA binding specificity of MLL-AF4 in leukemia"
collection: publications
permalink: /publication/2024-06-01-mll-af4-binding
date: 2024-06-01
venue: 'Genome Regulation and Cellular Fates in Homeostasis and Disease, Madrid, Spain'
category: posters
slidesurl: 'https://cchahrour.github.io/files/Gen_reg_poster_Catherine.pdf'
---
